In [1]:
import pandas as pd
import scanpy as sc
import numpy as np

jiaming_data_full = pd.read_csv("mutation_data/full_union_patient_gene_matrix.csv", index_col=0)
jiaming_data_colinear = pd.read_csv("mutation_data/colinear_union_patient_gene_matrix.csv", index_col=0)
jiaming_data_intersect = pd.read_csv("mutation_data/intersect_patient_gene_matrix.csv", index_col=0)
haowen_data = sc.read_h5ad("LUAD_Raw_counts_3yr_label_raw.h5ad")

### Sample Overlapping

In [2]:
haowen_samples = set(haowen_data.obs_names)
jiaming_samples = set(jiaming_data_full.index)

overlap_samples = haowen_samples.intersection(jiaming_samples)

haowen_data = haowen_data[haowen_data.obs_names.isin(overlap_samples)]
jiaming_data_full = jiaming_data_full.loc[list(overlap_samples)]

### Feature Overlapping

In [3]:
jiaming_genes = set(jiaming_data_full.columns)
haowen_genes = set(haowen_data.var_names)
km_genes = haowen_data.var_names[haowen_data.var["KM_markers"]]

overlap_genes = list(jiaming_genes.intersection(haowen_genes))
overlap_genes_with_km = list(set(overlap_genes).union(km_genes))

jiaming_data_full = jiaming_data_full[overlap_genes]
jiaming_data_colinear = jiaming_data_colinear[list(set(jiaming_data_colinear.columns).intersection(overlap_genes))]
jiaming_data_intersect = jiaming_data_intersect[list(set(jiaming_data_intersect.columns).intersection(overlap_genes))]

haowen_data = haowen_data[:, overlap_genes_with_km]
haowen_data.obs = haowen_data.obs[["survival_3yr_label"]]

haowen_data.var["jm_full_genes"] = [gene in jiaming_data_full.columns for gene in haowen_data.var_names]
haowen_data.var["jm_colinear_genes"] = [gene in jiaming_data_colinear.columns for gene in haowen_data.var_names]
haowen_data.var["jm_intersect_genes"] = [gene in jiaming_data_intersect.columns for gene in haowen_data.var_names]

### Merge data

In [6]:
haowen_data.uns["jiaming_data_full"] = jiaming_data_full
haowen_data.uns["jiaming_data_colinear"] = jiaming_data_colinear
haowen_data.uns["jiaming_data_intersect"] = jiaming_data_intersect

In [8]:
# Save data
haowen_data.write_h5ad("combined_data.h5ad")

In [9]:
haowen_data

AnnData object with n_obs × n_vars = 504 × 914
    obs: 'survival_3yr_label'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'KM_markers', 'jm_full_genes', 'jm_colinear_genes', 'jm_intersect_genes'
    uns: 'jiaming_data_full', 'jiaming_data_colinear', 'jiaming_data_intersect'
    layers: 'counts'

In [10]:
loopup_dict = {
    "alive": 1,
    "dead": 0
}

survival_3yr_label
0.0    134
1.0    112
Name: count, dtype: int64